# BP2 Gate 2 — Data Verification & Taxonomy Engineering
**Customer360 Navigator Enterprise Suite — Customer Friction Classification**

## Why this notebook exists
BP2 Gate 1's real run (2026-09-22) live-enumerated the actual `Company response to consumer` (6
distinct values) and `Timely response?` (2 distinct values) label strings for the first time in
this project's documentation, and deliberately deferred the bucket-to-severity-class mapping to
this Gate 2 notebook — the same sequencing BP1 used for its CFPB<->BANKING77 crosswalk (Gate 1
defines the target shape, Gate 2 builds the actual taxonomy against real, confirmed values).

## What this notebook does
1. Re-verifies live (never trusts a baked-in count without re-measuring — Master Plan zero-
   fabrication rule) that `Company response to consumer` and `Timely response?` still match the
   real values `configs/bp2_friction_severity_taxonomy.yaml` was authored against.
2. Runs a live crosstab between those two real fields — the severity taxonomy's precedence rule
   (`Timely response? == "No"` overrides `Company response to consumer`) is a documented judgment
   call, not an assumption that the two fields always agree, so this notebook checks rather than
   assumes.
3. Applies the documented severity taxonomy (`configs/bp2_friction_severity_taxonomy.yaml`,
   `src/taxonomy/friction_severity_mapper.py`) to produce `friction_severity_class` per real row —
   4 ordinal classes for genuinely resolved complaints (`LOW_FRICTION` … `HIGH_FRICTION`) plus 2
   excluded/non-ordinal classes (`EXCLUDED_PENDING` for the real 22.11% still "In progress",
   `EXCLUDED_UNKNOWN` for 2 null-response rows) — never silently folded into the ordinal scale.
4. Attaches `common_taxonomy_bucket` by reusing BP1 Gate 2's own crosswalk
   (`configs/taxonomy_mapping.yaml`, `taxonomy_mapper.cfpb_bucket_expr`) unmodified — this is how
   BP2 satisfies the Master Plan's "Integrates BANKING77: YES" requirement, since BANKING77 itself
   has no friction/severity field to train on directly.
5. Writes the BP2 CFPB-with-severity Gold layer to Parquet (WARP: Parquet over CSV for reused
   data) — every real row is tagged, none dropped here; filtering to the trainable subset
   (excluding `EXCLUDED_*` classes) is a Gate 3 decision, recorded here only as a real count.

## Standing rules this notebook follows
- **Execution boundary**: Claude wrote this notebook; it does not run it. You run it on your own
  machine, and the real, live-checked results below become this project's Gate 2 taxonomy record.
- **Zero-fabrication**: every distribution, crosstab, and row count below is computed live against
  the real CFPB file, not asserted from `policy.json` or this config without re-measuring.
- **HYPER**: reuses BP1 Gate 2's crosswalk (`taxonomy_mapper.py`) and BP1's config-sync helpers
  (`bp1_config_sync.py`) unmodified; the new severity logic lives in its own sibling module
  (`friction_severity_mapper.py`) rather than being bolted onto BP1's already-confirmed file.

In [ ]:
"""
Customer360 Navigator Enterprise Suite - BP2 Gate 2 data verification / taxonomy engineering
notebook. Single consolidated code cell (platform convention). Idempotent - safe to re-run.
"""

import os, sys, json, warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# ============================================================
# SECTION 1: Project root resolution (PROJECT_STRUCTURE_LOCKED.md rule #3)
# ============================================================
def _find_project_root() -> Path:
    marker = "PROJECT_STRUCTURE_LOCKED.md"
    env_override = os.environ.get("C360_PROJECT_ROOT")
    if env_override:
        if (Path(env_override) / marker).exists():
            return Path(env_override)
        raise RuntimeError(
            f"C360_PROJECT_ROOT is set to {env_override!r} but {marker} was not found there. "
            "Fix the environment variable rather than removing this check."
        )

    start = Path.cwd()
    cur = start
    for _ in range(8):
        if (cur / marker).exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent

    for depth_root, dirnames, filenames in os.walk(start):
        rel_depth = len(Path(depth_root).relative_to(start).parts)
        if rel_depth > 3:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]
        if marker in filenames:
            return Path(depth_root)

    raise RuntimeError(
        f"Could not resolve PROJECT_ROOT: no {marker} found by walking up from {start}, nor by "
        "searching up to 3 levels below it. Fix: add a cell at the TOP of this notebook (before "
        "this cell runs) with:\n"
        '    import os; os.environ["C360_PROJECT_ROOT"] = r"C:\\Users\\rnand\\Documents\\'
        'Customer360_Navigator_Enterprise_Suite"\n'
        "then re-run from the top."
    )

PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"[OK] Project root resolved: {PROJECT_ROOT.name}")

# ============================================================
# SECTION 2: WARP performance configuration - FIRST, before any heavy import
# ============================================================
from utils.performance_setup import configure_performance, memory_headroom_gb

perf_summary = configure_performance(project_root=PROJECT_ROOT)
print(f"[WARP] Headroom before heavy work: {memory_headroom_gb()} GB")

# ============================================================
# SECTION 3: Heavy imports (only after WARP configuration)
# ============================================================
import polars as pl
from IPython.display import display

from taxonomy.taxonomy_mapper import CFPB_DTYPES, load_mapping_config
from taxonomy.friction_severity_mapper import (
    load_severity_config,
    load_cfpb_with_severity,
    severity_crosstab_report,
    severity_distribution_report,
    build_bp2_severity_gold_layer,
)

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
CONFIGS_DIR = PROJECT_ROOT / "configs"
ARTIFACTS_DIR = PROJECT_ROOT / "notebooks" / "bp2_customer_friction_classification" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

CFPB_PATH = DATA_RAW / "cfpb_complaints.csv"
SEVERITY_CONFIG_PATH = CONFIGS_DIR / "bp2_friction_severity_taxonomy.yaml"
TAXONOMY_MAPPING_PATH = CONFIGS_DIR / "taxonomy_mapping.yaml"

for p in (CFPB_PATH, SEVERITY_CONFIG_PATH, TAXONOMY_MAPPING_PATH):
    if not p.exists():
        raise FileNotFoundError(
            f"Required input not found: {p}. Confirm BP2 Gate 1 and BP1 Gate 2 both completed "
            "(the severity taxonomy config and the CFPB<->BANKING77 crosswalk are both prerequisites)."
        )

severity_config = load_severity_config(SEVERITY_CONFIG_PATH)
taxonomy_mapping = load_mapping_config(TAXONOMY_MAPPING_PATH)
print(f"[OK] Loaded bp2_friction_severity_taxonomy.yaml: "
      f"{len(severity_config['precedence_order'])} precedence rules, "
      f"{len(severity_config['severity_classes'])} ordinal classes, "
      f"{len(severity_config['excluded_classes'])} excluded classes.")
print(f"[OK] Loaded taxonomy_mapping.yaml (BP1 Gate 2, reused unmodified): "
      f"{len(taxonomy_mapping['banking77_category_to_bucket'])} BANKING77 categories mapped.")

# ============================================================
# SECTION 4: LIVE drift check - re-measure Company response to consumer / Timely response?
# against what bp2_friction_severity_taxonomy.yaml was authored against. Zero-fabrication: never
# trust the config's baked-in counts without re-measuring against the real file.
# ============================================================
live_company_response_counts = (
    pl.scan_csv(CFPB_PATH, schema_overrides=CFPB_DTYPES)
    .group_by("Company response to consumer")
    .agg(pl.len().alias("row_count"))
    .collect()
)
live_timely_counts = (
    pl.scan_csv(CFPB_PATH, schema_overrides=CFPB_DTYPES)
    .group_by("Timely response?")
    .agg(pl.len().alias("row_count"))
    .collect()
)

live_company_response_dict = dict(zip(
    live_company_response_counts["Company response to consumer"].cast(pl.Utf8).to_list(),
    live_company_response_counts["row_count"].to_list(),
))
live_timely_dict = dict(zip(
    live_timely_counts["Timely response?"].cast(pl.Utf8).to_list(),
    live_timely_counts["row_count"].to_list(),
))
documented_company_response_dict = {
    d["value"]: d["row_count"] for d in severity_config["company_response_to_consumer_distribution"]
}
documented_timely_dict = {
    d["value"]: d["row_count"] for d in severity_config["timely_response_distribution"]
}

company_response_drift = {
    k: {"documented": documented_company_response_dict.get(k), "live": live_company_response_dict.get(k)}
    for k in set(documented_company_response_dict) | set(live_company_response_dict)
    if documented_company_response_dict.get(k) != live_company_response_dict.get(k)
}
timely_drift = {
    k: {"documented": documented_timely_dict.get(k), "live": live_timely_dict.get(k)}
    for k in set(documented_timely_dict) | set(live_timely_dict)
    if documented_timely_dict.get(k) != live_timely_dict.get(k)
}

if company_response_drift or timely_drift:
    print(f"[DRIFT DETECTED] company_response_drift={json.dumps(company_response_drift, indent=2)} "
          f"timely_drift={json.dumps(timely_drift, indent=2)}")
else:
    print("[OK] Live 'Company response to consumer' and 'Timely response?' distributions match "
          "bp2_friction_severity_taxonomy.yaml exactly - no drift since BP2 Gate 1's real run.")

# ============================================================
# SECTION 5: Live crosstab - does 'Untimely response' (Company response to consumer) ever
# disagree with Timely response?=='No' on the same row? The severity taxonomy's precedence rule
# checks Timely response? first; this crosstab confirms (rather than assumes) how the two real
# fields relate before that precedence choice is treated as settled.
# ============================================================
cfpb_lazy_base = pl.scan_csv(CFPB_PATH, schema_overrides=CFPB_DTYPES)
crosstab = severity_crosstab_report(cfpb_lazy_base)
print("\n=== LIVE CROSSTAB: Company response to consumer x Timely response? ===")
display(crosstab.to_pandas())

untimely_label_rows = crosstab.filter(pl.col("Company response to consumer") == "Untimely response")["row_count"].sum()
untimely_label_and_no = crosstab.filter(
    (pl.col("Company response to consumer") == "Untimely response") & (pl.col("Timely response?") == "No")
)["row_count"].sum()
print(f"\n[FINDING] Of {untimely_label_rows} rows labeled 'Untimely response', "
      f"{untimely_label_and_no} also have Timely response?=='No' "
      f"({untimely_label_and_no / untimely_label_rows:.1%} agreement) - the remainder are resolved "
      "by the precedence rule's Timely response? check running first, which the config's "
      "precedence_order documents explicitly as a judgment call, not an assumed redundancy.")

# ============================================================
# SECTION 6: Apply the severity taxonomy + BP1's taxonomy-bucket crosswalk (BANKING77 integration)
# ============================================================
cfpb_lazy = load_cfpb_with_severity(CFPB_PATH, severity_config, taxonomy_mapping=taxonomy_mapping)

severity_dist = severity_distribution_report(cfpb_lazy)
print("\n=== FRICTION SEVERITY CLASS DISTRIBUTION (real, live-computed) ===")
display(severity_dist.to_pandas())

severity_dist_path = ARTIFACTS_DIR / "gate2_severity_distribution.csv"
severity_dist.write_csv(severity_dist_path)
print(f"[SAVED] {severity_dist_path.relative_to(PROJECT_ROOT)}")

ordinal_classes = set(severity_config["severity_classes"].keys())
excluded_classes = set(severity_config["excluded_classes"].keys())
observed_classes = set(severity_dist["friction_severity_class"].to_list())
unexpected_classes = observed_classes - ordinal_classes - excluded_classes

trainable_rows = int(
    severity_dist.filter(pl.col("friction_severity_class").is_in(list(ordinal_classes)))["row_count"].sum()
)
excluded_rows = int(
    severity_dist.filter(pl.col("friction_severity_class").is_in(list(excluded_classes)))["row_count"].sum()
)
print(f"\n[FINDING] Real trainable rows (4 ordinal severity classes): {trainable_rows:,} "
      f"({trainable_rows / severity_config['total_cfpb_rows']:.2%} of the full extract). "
      f"Excluded rows (EXCLUDED_PENDING + EXCLUDED_UNKNOWN): {excluded_rows:,} "
      f"({excluded_rows / severity_config['total_cfpb_rows']:.2%}) - a Gate 3 decision, not "
      "dropped from the Gold layer written below, only flagged as non-trainable here.")

# ============================================================
# SECTION 7: BANKING77-overlap coverage on the severity-tagged rows (real, live)
# ============================================================
bucket_coverage = (
    cfpb_lazy.group_by("common_taxonomy_bucket")
    .agg(pl.len().alias("row_count"))
    .collect()
    .sort("row_count", descending=True)
)
in_scope_rows = int(
    bucket_coverage.filter(
        ~pl.col("common_taxonomy_bucket").is_in(["OUT_OF_SCOPE_NO_BANKING77_OVERLAP", "UNMAPPED_UNKNOWN_PRODUCT"])
    )["row_count"].sum()
)
print(f"\n[FINDING] Rows with a real BANKING77-overlap taxonomy bucket (reused from BP1 Gate 2, "
      f"unmodified): {in_scope_rows:,} ({in_scope_rows / severity_config['total_cfpb_rows']:.2%}) - "
      "this is the feature BP2 uses to satisfy 'Integrates BANKING77: YES', not a friction/severity "
      "signal from BANKING77 itself (which does not exist).")

# ============================================================
# SECTION 8: Write the BP2 CFPB-with-severity Gold layer (Parquet, WARP)
# ============================================================
gold_summary = build_bp2_severity_gold_layer(cfpb_lazy, DATA_PROCESSED)
print(f"\n[SAVED] BP2 CFPB Severity Gold: {gold_summary['cfpb_severity_gold_path']} "
      f"({gold_summary['cfpb_severity_gold_rows_written']} rows)")

# ============================================================
# SECTION 9: Write the Gate 2 config block (marker-based, order-independent - reuses BP1's
# bp1_config_sync.py unmodified, per LESSONS_LEARNED_APPLIED.md #20)
# ============================================================
from utils.bp1_config_sync import write_gate_block  # noqa: E402

bp2_config_path = CONFIGS_DIR / "bp2_customer_friction_classification.yaml"
gate2_marker = "# --- Gate 2 (Data Verification & Taxonomy Engineering) results (appended, idempotent overwrite) ---"
gate2_block_lines = [
    f"severity_taxonomy_config: \"{SEVERITY_CONFIG_PATH.relative_to(PROJECT_ROOT).as_posix()}\"",
    f"drift_vs_gate1_live_check: {'none' if not (company_response_drift or timely_drift) else 'DRIFT_DETECTED'}",
    f"untimely_response_label_vs_timely_no_agreement_rows: {int(untimely_label_and_no)}",
    f"untimely_response_label_total_rows: {int(untimely_label_rows)}",
    "severity_class_row_counts:",
] + [
    f"  {row['friction_severity_class']}: {int(row['row_count'])}"
    for row in severity_dist.to_dicts()
] + [
    f"trainable_rows_4_ordinal_classes: {trainable_rows}",
    f"excluded_rows_pending_and_unknown: {excluded_rows}",
    f"banking77_taxonomy_bucket_in_scope_rows: {in_scope_rows}",
    f"cfpb_severity_gold_path: \"{Path(gold_summary['cfpb_severity_gold_path']).relative_to(PROJECT_ROOT).as_posix()}\"",
    f"cfpb_severity_gold_rows_written: {gold_summary['cfpb_severity_gold_rows_written']}",
]
write_gate_block(bp2_config_path, gate2_marker, gate2_block_lines)
print(f"[SAVED] gate2 block written to {bp2_config_path.relative_to(PROJECT_ROOT)}")

# ============================================================
# SECTION 10: Structural integrity checks - raise AssertionError, never silently pass
# ============================================================
checks = {
    "no_drift_vs_gate1_documented_distributions": not (company_response_drift or timely_drift),
    "no_unexpected_severity_class_values": len(unexpected_classes) == 0,
    "severity_distribution_sums_to_total_rows": int(severity_dist["row_count"].sum()) == severity_config["total_cfpb_rows"],
    "gold_layer_row_count_matches_source": gold_summary["cfpb_severity_gold_rows_written"] == severity_config["total_cfpb_rows"],
    "trainable_plus_excluded_equals_total": trainable_rows + excluded_rows == severity_config["total_cfpb_rows"],
    "severity_distribution_csv_written": severity_dist_path.exists(),
    "bp2_config_gate2_block_written": bp2_config_path.exists(),
}

print("\n=== INTEGRITY CHECKS ===")
for name, passed in checks.items():
    status = "[PASS]" if passed else "[FAIL]"
    print(f"{status} {name}")
    assert passed, f"[CHECK FAILED] {name}"

print("\n[ALL CHECKS PASSED] BP2 Gate 2 complete - friction severity taxonomy applied to every "
      f"real row, {trainable_rows:,} rows trainable across 4 ordinal classes, "
      f"{excluded_rows:,} rows excluded (pending/unknown) and flagged not dropped. "
      "Proceed to BP2 Gate 3 (Model Benchmark) next.")